# Phase 3: Big Data Pipeline + GCP
**Environment:** Local (PySpark local mode)

**Cloud:** Google Cloud Platform (GCS + Cloud SQL)

**Goal:** Build scalable data pipeline to process waste annotation data and store results in GCS (Parquet) and Cloud SQL (PostgreSQL)

## 1. Setup PySpark
Initialize Spark session in local mode.

local[*] uses all available CPU cores.

shuffle.partitions set to 4 for small dataset efficiency.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, DateType, TimestampType, BooleanType
)

# Initialize Spark session (local mode)
spark = SparkSession.builder \
    .appName('WasteIntelligencePipeline') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version : {spark.version}')
print(f'Master        : {spark.sparkContext.master}')

Spark version : 3.5.1
Master        : local[*]


## 2. Define Schema
Define explicit schema for raw annotation CSV.

Never infer schema in production — explicit schema catches type errors early and improves read performance significantly.

In [2]:
# Define explicit schema — never infer from CSV in production
RAW_SCHEMA = StructType([
    StructField('annotation_id', IntegerType(), False),
    StructField('image_id',      IntegerType(), False),
    StructField('filename',      StringType(),  False),
    StructField('split',         StringType(),  False),
    StructField('class_name',    StringType(),  False),
    StructField('bbox_x',        FloatType(),   True),
    StructField('bbox_y',        FloatType(),   True),
    StructField('bbox_width',    FloatType(),   True),
    StructField('bbox_height',   FloatType(),   True),
    StructField('bbox_area',     FloatType(),   False),
    StructField('image_width',   IntegerType(), True),
    StructField('image_height',  IntegerType(), True),
    StructField('datetime',      TimestampType(), True),
    StructField('date',          DateType(),    True),
    StructField('hour',          IntegerType(), True),
])

print('Schema defined:')
print(RAW_SCHEMA.simpleString())

Schema defined:
struct<annotation_id:int,image_id:int,filename:string,split:string,class_name:string,bbox_x:float,bbox_y:float,bbox_width:float,bbox_height:float,bbox_area:float,image_width:int,image_height:int,datetime:timestamp,date:date,hour:int>


## 3. Ingest Stage
Load raw annotations CSV into Spark DataFrame using defined schema.

This is the entry point of the pipeline.

Verify record count matches Phase 1 EDA findings (20,398 records).

In [3]:
# Load raw CSV into Spark DataFrame
RAW_PATH = '../data/processed/annotations_processed.csv'

df_raw = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv(RAW_PATH)

# Cast types manually after load
from pyspark.sql.types import IntegerType, FloatType, DateType, TimestampType

df_raw = df_raw \
    .withColumn('annotation_id', F.col('annotation_id').cast(IntegerType())) \
    .withColumn('image_id',      F.col('image_id').cast(IntegerType())) \
    .withColumn('bbox_x',        F.col('bbox_x').cast(FloatType())) \
    .withColumn('bbox_y',        F.col('bbox_y').cast(FloatType())) \
    .withColumn('bbox_width',    F.col('bbox_width').cast(FloatType())) \
    .withColumn('bbox_height',   F.col('bbox_height').cast(FloatType())) \
    .withColumn('bbox_area',     F.col('bbox_area').cast(FloatType())) \
    .withColumn('image_width',   F.col('image_width').cast(IntegerType())) \
    .withColumn('image_height',  F.col('image_height').cast(IntegerType())) \
    .withColumn('hour',          F.col('hour').cast(IntegerType())) \
    .withColumn('date',          F.to_date(F.col('date'), 'yyyy-MM-dd')) \
    .withColumn('datetime',      F.to_timestamp(F.col('datetime'), 'yyyy-MM-dd HH:mm:ss'))

print('=== Ingest Stage ===')
print(f'Total records : {df_raw.count()}')
print(f'Partitions    : {df_raw.rdd.getNumPartitions()}')
print('\nSchema:')
df_raw.printSchema()
print('\nSample rows:')
df_raw.show(5, truncate=True)

=== Ingest Stage ===
Total records : 20398
Partitions    : 1

Schema:
root
 |-- annotation_id: integer (nullable = true)
 |-- image_id: integer (nullable = true)
 |-- filename: string (nullable = true)
 |-- split: string (nullable = true)
 |-- class_name: string (nullable = true)
 |-- bbox_x: float (nullable = true)
 |-- bbox_y: float (nullable = true)
 |-- bbox_width: float (nullable = true)
 |-- bbox_height: float (nullable = true)
 |-- bbox_area: float (nullable = true)
 |-- image_width: integer (nullable = true)
 |-- image_height: integer (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)


Sample rows:
+-------------+--------+--------------------+-----+----------+------+------+----------+-----------+---------+-----------+------------+-------------------+----------+----+
|annotation_id|image_id|            filename|split|class_name|bbox_x|bbox_y|bbox_width|bbox_height|bbox_area|image_width|image_height

## 4. Data Quality Check
Check for null values in critical columns and verify data integrity.

Cross-reference class distribution and date range against Phase 1 EDA results to confirm ingestion is correct.

In [4]:
# Check for nulls in critical columns
print('=== Data Quality Check ===\n')

critical_cols = ['annotation_id', 'class_name', 'bbox_area', 'date']

for col in critical_cols:
    null_count = df_raw.filter(F.col(col).isNull()).count()
    print(f'  {col:<20} nulls: {null_count}')

# Check class distribution
print('\nClass distribution:')
df_raw.groupBy('class_name') \
      .count() \
      .orderBy('count', ascending=False) \
      .show()

# Check date range
print('Date range:')
df_raw.select(
    F.min('date').alias('earliest'),
    F.max('date').alias('latest'),
    F.countDistinct('date').alias('unique_days')
).show()

# Check split distribution
print('Split distribution:')
df_raw.groupBy('split').count().show()

=== Data Quality Check ===

  annotation_id        nulls: 0
  class_name           nulls: 0
  bbox_area            nulls: 0
  date                 nulls: 0

Class distribution:
+---------------+-----+
|     class_name|count|
+---------------+-----+
|        Plastic| 8479|
|Paper-Cardboard| 6994|
|    Mixed Waste| 2587|
|          Metal| 1447|
|           Wood|  891|
+---------------+-----+

Date range:
+----------+----------+-----------+
|  earliest|    latest|unique_days|
+----------+----------+-----------+
|2024-08-07|2024-10-01|         32|
+----------+----------+-----------+

Split distribution:
+-----+-----+
|split|count|
+-----+-----+
| test| 1039|
|train|17988|
|valid| 1371|
+-----+-----+



## 5. Register as Spark SQL Table
Register DataFrame as a temporary view to enable Spark SQL queries.

Test with a class summary query to verify SQL interface works correctly before moving to transform stage.

In [5]:
# Register as temp view for Spark SQL queries
df_raw.createOrReplaceTempView('raw_annotations')

# Test SQL query
result = spark.sql("""
    SELECT
        class_name,
        COUNT(*)        AS annotation_count,
        ROUND(AVG(bbox_area), 2) AS avg_bbox_area,
        ROUND(MIN(bbox_area), 2) AS min_bbox_area,
        ROUND(MAX(bbox_area), 2) AS max_bbox_area
    FROM raw_annotations
    GROUP BY class_name
    ORDER BY annotation_count DESC
""")

print('=== Spark SQL — Class Summary ===')
result.show()

=== Spark SQL — Class Summary ===
+---------------+----------------+-------------+-------------+-------------+
|     class_name|annotation_count|avg_bbox_area|min_bbox_area|max_bbox_area|
+---------------+----------------+-------------+-------------+-------------+
|        Plastic|            8479|       2470.5|         1.03|    111487.91|
|Paper-Cardboard|            6994|      2978.26|         1.08|     144293.0|
|    Mixed Waste|            2587|     27221.77|         1.01|    248217.73|
|          Metal|            1447|      3697.67|         45.0|     12072.98|
|           Wood|             891|      3793.76|         36.6|    256540.19|
+---------------+----------------+-------------+-------------+-------------+



## 6. Transform Stage — Clean & Standardize
Clean data, handle edge cases, and standardize format before aggregation. Apply business rules from Phase 1 findings.

In [6]:
# Transform stage — clean and standardize
df_clean = df_raw \
    .filter(F.col('bbox_area') >= 1.0) \
    .filter(F.col('class_name') != 'wastes') \
    .filter(F.col('date').isNotNull()) \
    .withColumn('bbox_area_log', F.log1p(F.col('bbox_area'))) \
    .withColumn('bbox_area_pct', F.col('bbox_area') / (F.col('image_width') * F.col('image_height'))) \
    .withColumn('time_of_day', F.when(F.col('hour').between(6, 11),  'morning')
                                .when(F.col('hour').between(12, 17), 'afternoon')
                                .when(F.col('hour').between(18, 23), 'evening')
                                .otherwise('night')) \
    .withColumn('month', F.month(F.col('date'))) \
    .withColumn('week_of_year', F.weekofyear(F.col('date'))) \
    .withColumn('day_of_week', F.dayofweek(F.col('date')))

print('=== Transform Stage ===')
print(f'Records before clean : {df_raw.count()}')
print(f'Records after clean  : {df_clean.count()}')
print(f'Removed              : {df_raw.count() - df_clean.count()}')
print('\nNew columns added:')
for col in ['bbox_area_log', 'bbox_area_pct', 'time_of_day', 'month', 'week_of_year', 'day_of_week']:
    print(f'  {col}')
df_clean.show(3, truncate=True)

=== Transform Stage ===
Records before clean : 20398
Records after clean  : 20398
Removed              : 0

New columns added:
  bbox_area_log
  bbox_area_pct
  time_of_day
  month
  week_of_year
  day_of_week
+-------------+--------+--------------------+-----+----------+------+------+----------+-----------+---------+-----------+------------+-------------------+----------+----+-----------------+--------------------+-----------+-----+------------+-----------+
|annotation_id|image_id|            filename|split|class_name|bbox_x|bbox_y|bbox_width|bbox_height|bbox_area|image_width|image_height|           datetime|      date|hour|    bbox_area_log|       bbox_area_pct|time_of_day|month|week_of_year|day_of_week|
+-------------+--------+--------------------+-----+----------+------+------+----------+-----------+---------+-----------+------------+-------------------+----------+----+-----------------+--------------------+-----------+-----+------------+-----------+
|            1|       0|2024-09

## 7. Validate Transform Output
Verify transformed data quality and new column values are correct.

In [7]:
# Validate time_of_day distribution
print('Time of day distribution:')
df_clean.groupBy('time_of_day').count().orderBy('count', ascending=False).show()

# Validate month distribution
print('Month distribution:')
df_clean.groupBy('month').count().orderBy('month').show()

# Validate bbox_area_pct range (should be 0-1)
print('bbox_area_pct range:')
df_clean.select(
    F.min('bbox_area_pct').alias('min'),
    F.max('bbox_area_pct').alias('max'),
    F.avg('bbox_area_pct').alias('mean')
).show()

Time of day distribution:
+-----------+-----+
|time_of_day|count|
+-----------+-----+
|  afternoon|16790|
|    morning| 2572|
|    evening|  865|
|      night|  171|
+-----------+-----+

Month distribution:
+-----+-----+
|month|count|
+-----+-----+
|    8| 3312|
|    9|16171|
|   10|  915|
+-----+-----+

bbox_area_pct range:
+--------------------+------------------+--------------------+
|                 min|               max|                mean|
+--------------------+------------------+--------------------+
|2.465820289216935...|0.6263188171386719|0.014474036120603844|
+--------------------+------------------+--------------------+



## 8. Register Transformed Data as Spark SQL View
Register cleaned DataFrame for Spark SQL aggregation queries.

In [8]:
df_clean.createOrReplaceTempView('clean_annotations')

# Quick verification
count = spark.sql('SELECT COUNT(*) as total FROM clean_annotations').collect()[0]['total']
classes = spark.sql('SELECT DISTINCT class_name FROM clean_annotations ORDER BY class_name').collect()

print(f'clean_annotations registered: {count} records')
print(f'Classes: {[r["class_name"] for r in classes]}')

clean_annotations registered: 20398 records
Classes: ['Metal', 'Mixed Waste', 'Paper-Cardboard', 'Plastic', 'Wood']


## 9. Aggregation Stage — Daily Summary
Calculate daily statistics per waste class using Spark SQL.

Output will be stored in Cloud SQL daily_waste_summary table.

In [9]:
# Daily aggregation using Spark SQL
df_daily = spark.sql("""
    SELECT
        date,
        class_name,
        COUNT(*)                    AS annotation_count,
        ROUND(SUM(bbox_area), 2)    AS total_bbox_area,
        ROUND(AVG(bbox_area), 2)    AS mean_bbox_area,
        ROUND(STDDEV(bbox_area), 2) AS std_bbox_area,
        ROUND(MIN(bbox_area), 2)    AS min_bbox_area,
        ROUND(MAX(bbox_area), 2)    AS max_bbox_area,
        COUNT(DISTINCT image_id)    AS image_count
    FROM clean_annotations
    GROUP BY date, class_name
    ORDER BY date, class_name
""")

print('=== Daily Aggregation ===')
print(f'Total rows : {df_daily.count()}')
print('\nSample (first 10 rows):')
df_daily.show(10, truncate=False)

=== Daily Aggregation ===
Total rows : 114

Sample (first 10 rows):
+----------+---------------+----------------+---------------+--------------+-------------+-------------+-------------+-----------+
|date      |class_name     |annotation_count|total_bbox_area|mean_bbox_area|std_bbox_area|min_bbox_area|max_bbox_area|image_count|
+----------+---------------+----------------+---------------+--------------+-------------+-------------+-------------+-----------+
|2024-08-07|Mixed Waste    |87              |2950845.9      |33917.77      |40531.94     |572.0        |140650.0     |20         |
|2024-08-07|Paper-Cardboard|83              |661571.64      |7970.74       |6101.83      |391.0        |21740.61     |9          |
|2024-08-07|Plastic        |55              |122742.58      |2231.68       |2120.51      |196.0        |12688.0      |9          |
|2024-08-08|Mixed Waste    |60              |2360145.15     |39335.75      |52094.17     |377.0        |213152.7     |20         |
|2024-08-08|Pap

## 10. Aggregation Stage — Weekly Summary
Calculate weekly statistics per waste class.

Output will be stored in Cloud SQL weekly_waste_summary table.

In [10]:
# Weekly aggregation
df_weekly = spark.sql("""
    SELECT
        DATE_TRUNC('week', date)    AS week_start,
        class_name,
        COUNT(*)                    AS annotation_count,
        ROUND(SUM(bbox_area), 2)    AS total_bbox_area,
        ROUND(AVG(bbox_area), 2)    AS mean_bbox_area,
        COUNT(DISTINCT image_id)    AS image_count
    FROM clean_annotations
    GROUP BY DATE_TRUNC('week', date), class_name
    ORDER BY week_start, class_name
""")

print('=== Weekly Aggregation ===')
print(f'Total rows : {df_weekly.count()}')
df_weekly.show(15, truncate=False)

=== Weekly Aggregation ===
Total rows : 37
+-------------------+---------------+----------------+---------------+--------------+-----------+
|week_start         |class_name     |annotation_count|total_bbox_area|mean_bbox_area|image_count|
+-------------------+---------------+----------------+---------------+--------------+-----------+
|2024-08-05 00:00:00|Mixed Waste    |259             |9483653.24     |36616.42      |80         |
|2024-08-05 00:00:00|Paper-Cardboard|415             |1707616.85     |4114.74       |40         |
|2024-08-05 00:00:00|Plastic        |371             |1238935.69     |3339.45       |40         |
|2024-08-05 00:00:00|Wood           |187             |1743573.72     |9323.92       |30         |
|2024-08-12 00:00:00|Mixed Waste    |68              |3999693.58     |58819.02      |45         |
|2024-08-12 00:00:00|Paper-Cardboard|139             |558642.1       |4019.01       |29         |
|2024-08-12 00:00:00|Plastic        |154             |398038.19      |2584.

## 11. Aggregation Stage — Class-level Summary
Overall statistics per class across entire dataset.

Used for dashboard overview and LLM context in Phase 5.

In [11]:
# Overall class summary
df_class_summary = spark.sql("""
    SELECT
        class_name,
        COUNT(*)                       AS total_annotations,
        COUNT(DISTINCT image_id)        AS total_images,
        COUNT(DISTINCT date)            AS active_days,
        ROUND(AVG(bbox_area), 2)        AS mean_bbox_area,
        ROUND(STDDEV(bbox_area), 2)     AS std_bbox_area,
        ROUND(MIN(bbox_area), 2)        AS min_bbox_area,
        ROUND(MAX(bbox_area), 2)        AS max_bbox_area,
        ROUND(AVG(bbox_area_pct), 4)    AS mean_area_pct
    FROM clean_annotations
    GROUP BY class_name
    ORDER BY total_annotations DESC
""")

print('=== Class-level Summary ===')
df_class_summary.show(truncate=False)

=== Class-level Summary ===
+---------------+-----------------+------------+-----------+--------------+-------------+-------------+-------------+-------------+
|class_name     |total_annotations|total_images|active_days|mean_bbox_area|std_bbox_area|min_bbox_area|max_bbox_area|mean_area_pct|
+---------------+-----------------+------------+-----------+--------------+-------------+-------------+-------------+-------------+
|Plastic        |8479             |548         |29         |2470.5        |2989.22      |1.03         |111487.91    |0.006        |
|Paper-Cardboard|6994             |508         |27         |2978.26       |3289.01      |1.08         |144293.0     |0.0073       |
|Mixed Waste    |2587             |880         |32         |27221.77      |38617.38     |1.01         |248217.73    |0.0665       |
|Metal          |1447             |57          |3          |3697.67       |2398.22      |45.0         |12072.98     |0.009        |
|Wood           |891              |276         |

## 12. Load Stage — Save Parquet to GCS
Save aggregated DataFrames as partitioned Parquet files to Google Cloud Storage.

Parquet format enables efficient columnar storage and fast querying.

In [13]:
from pathlib import Path
from google.cloud import storage

LOCAL_PROCESSED = '../data/processed'
BUCKET_NAME     = 'waste-intelligence-data'

# Save as CSV locally
Path(LOCAL_PROCESSED).mkdir(parents=True, exist_ok=True)

df_daily.toPandas().to_csv(f'{LOCAL_PROCESSED}/daily_summary.csv', index=False)
print('Daily summary saved locally')

df_weekly.toPandas().to_csv(f'{LOCAL_PROCESSED}/weekly_summary.csv', index=False)
print('Weekly summary saved locally')

df_class_summary.toPandas().to_csv(f'{LOCAL_PROCESSED}/class_summary.csv', index=False)
print('Class summary saved locally')

# Upload to GCS using google-cloud-storage
client = storage.Client(project='waste-intelligence')
bucket = client.bucket(BUCKET_NAME)

for fname in ['daily_summary.csv', 'weekly_summary.csv', 'class_summary.csv']:
    blob = bucket.blob(f'processed/{fname}')
    blob.upload_from_filename(f'{LOCAL_PROCESSED}/{fname}')
    print(f'  Uploaded: {fname}')

# Verify
print('\nGCS files:')
for blob in client.list_blobs(BUCKET_NAME, prefix='processed/'):
    print(f'  {blob.name}')

Daily summary saved locally
Weekly summary saved locally
Class summary saved locally


c:\Users\bhumi\Desktop\Kinsei_Sangyo\industrial-waste-intelligence\venv\lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


  Uploaded: daily_summary.csv
  Uploaded: weekly_summary.csv
  Uploaded: class_summary.csv

GCS files:
  processed/
  processed/annotations_processed.csv
  processed/class_summary.csv
  processed/daily_summary.csv
  processed/parquet/
  processed/weekly_summary.csv


## 13. Load Stage — Save to Cloud SQL
Load aggregated summary statistics into Cloud SQL PostgreSQL tables for dashboard queries and API serving layer.

In [15]:
import psycopg2
import pandas as pd

DB_CONFIG = {
    'host'    : '34.104.146.13',
    'database': 'waste_intelligence',
    'user'    : 'postgres',
    'password': '12345678',
    'port'    : 5432
}

def load_to_cloudsql(df_spark, table_name, db_config):
    """Convert Spark DataFrame to pandas and load into Cloud SQL."""
    df_pd = df_spark.toPandas()

    conn = psycopg2.connect(**db_config)
    cur  = conn.cursor()

    # Build insert query
    cols    = ', '.join(df_pd.columns)
    vals    = ', '.join(['%s'] * len(df_pd.columns))
    query   = f'INSERT INTO {table_name} ({cols}) VALUES ({vals}) ON CONFLICT DO NOTHING'

    records = [tuple(row) for row in df_pd.itertuples(index=False)]
    cur.executemany(query, records)
    conn.commit()

    print(f'Loaded {len(records)} rows into {table_name}')
    cur.close()
    conn.close()

# Load daily summary
load_to_cloudsql(df_daily, 'daily_waste_summary', DB_CONFIG)

# Load weekly summary — rename week_start column
df_weekly_renamed = df_weekly.withColumnRenamed('week_start', 'week_start')
load_to_cloudsql(df_weekly_renamed, 'weekly_waste_summary', DB_CONFIG)

# Load class summary — save as CSV locally for reference
df_class_summary.toPandas().to_csv('../data/processed/class_summary.csv', index=False)
print('Class summary saved locally')

Loaded 114 rows into daily_waste_summary
Loaded 37 rows into weekly_waste_summary
Class summary saved locally


## 14. Verify Pipeline End-to-End
Query Cloud SQL to confirm data was loaded correctly.

Cross-reference with Spark aggregation results.

In [16]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# Check daily_waste_summary
cur.execute('SELECT COUNT(*) FROM daily_waste_summary')
daily_count = cur.fetchone()[0]

# Check weekly_waste_summary
cur.execute('SELECT COUNT(*) FROM weekly_waste_summary')
weekly_count = cur.fetchone()[0]

# Sample query
cur.execute("""
    SELECT date, class_name, annotation_count, mean_bbox_area
    FROM daily_waste_summary
    ORDER BY date, class_name
    LIMIT 5
""")
rows = cur.fetchall()

print('=== Pipeline Verification ===')
print(f'daily_waste_summary  : {daily_count} rows')
print(f'weekly_waste_summary : {weekly_count} rows')
print('\nSample from Cloud SQL:')
for row in rows:
    print(f'  {row}')

cur.close()
conn.close()

=== Pipeline Verification ===
daily_waste_summary  : 114 rows
weekly_waste_summary : 37 rows

Sample from Cloud SQL:
  (datetime.date(2024, 8, 7), 'Mixed Waste', 87, 33917.77)
  (datetime.date(2024, 8, 7), 'Paper-Cardboard', 83, 7970.74)
  (datetime.date(2024, 8, 7), 'Plastic', 55, 2231.68)
  (datetime.date(2024, 8, 8), 'Mixed Waste', 60, 39335.75)
  (datetime.date(2024, 8, 8), 'Paper-Cardboard', 81, 2870.95)
